In [0]:
from pyspark.sql.functions import (
    col,
    count,
    avg,
    min,
    max,
    sum,
    round,
    desc
)

In [0]:
scd2_df = spark.table("silver_employee_scd2")

print("Total SCD2 Records:", scd2_df.count())

In [0]:
current_employees = (
    scd2_df
    .filter(col("is_current") == True)
)

print("Current Employees:", current_employees.count())

display(current_employees)

In [0]:
department_count = (
    current_employees
    .groupBy("department")
    .agg(
        count("*").alias("employee_count")
    )
    .orderBy(desc("employee_count"))
)

display(department_count)

In [0]:
salary_analysis = (
    current_employees
    .groupBy("department")
    .agg(
        count("*").alias("employee_count"),
        round(avg("salary"), 2).alias("average_salary"),
        min("salary").alias("minimum_salary"),
        max("salary").alias("maximum_salary")
    )
    .orderBy(desc("average_salary"))
)

display(salary_analysis)

In [0]:
status_analysis = (
    current_employees
    .groupBy("status")
    .agg(
        count("*").alias("employee_count")
    )
    .orderBy(desc("employee_count"))
)

display(status_analysis)

In [0]:
department_status = (
    current_employees
    .groupBy("department", "status")
    .agg(
        count("*").alias("employee_count")
    )
    .orderBy("department", "status")
)

display(department_status)

In [0]:
top_salary_employees = (
    current_employees
    .select(
        "emp_id",
        "name",
        "department",
        "salary",
        "status"
    )
    .orderBy(desc("salary"))
    .limit(10)
)

display(top_salary_employees)

In [0]:
current_employees.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_current_employees")

In [0]:
salary_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_department_salary")

In [0]:
department_count.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_department_count")

In [0]:
status_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_employee_status")

In [0]:
department_status.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_department_status")

In [0]:
top_salary_employees.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_top_salary_employees")

In [0]:
print("Current Employees:", spark.table("gold_current_employees").count())
print("Department Salary:", spark.table("gold_department_salary").count())
print("Department Count:", spark.table("gold_department_count").count())
print("Employee Status:", spark.table("gold_employee_status").count())
print("Department Status:", spark.table("gold_department_status").count())
print("Top Salary:", spark.table("gold_top_salary_employees").count())

In [0]:
from pyspark.sql.functions import datediff, current_date, round

tenure_analysis = (
    current_employees
    .withColumn(
        "tenure_years",
        round(
            datediff(
                current_date(),
                col("join_date")
            ) / 365.25,
            2
        )
    )
)

display(
    tenure_analysis.select(
        "emp_id",
        "name",
        "department",
        "join_date",
        "tenure_years"
    )
)

In [0]:
overall_tenure = (
    tenure_analysis
    .agg(
        round(
            avg("tenure_years"),
            2
        ).alias("average_tenure_years")
    )
)

display(overall_tenure)

In [0]:
department_tenure = (
    tenure_analysis
    .groupBy("department")
    .agg(
        count("*").alias("employee_count"),
        round(
            avg("tenure_years"),
            2
        ).alias("average_tenure_years")
    )
    .orderBy(desc("average_tenure_years"))
)

display(department_tenure)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window_spec = Window.partitionBy("emp_id").orderBy("effective_from_day")

status_history = (
    scd2_df
    .withColumn(
        "previous_status",
        lag("status").over(window_spec)
    )
)

display(
    status_history.select(
        "emp_id",
        "name",
        "status",
        "previous_status",
        "effective_from_day",
        "effective_to_day"
    )
)

In [0]:
attrition_events = (
    status_history
    .filter(
        (col("previous_status") == "Active") &
        (col("status") == "Inactive")
    )
)

print("Attrition Events:", attrition_events.count())

display(
    attrition_events.select(
        "emp_id",
        "name",
        "department",
        "status",
        "previous_status",
        "effective_from_day"
    )
)

In [0]:
display(
    scd2_df
    .groupBy("status")
    .count()
    .orderBy(desc("count"))
)

In [0]:
resignation_events = (
    status_history
    .filter(
        (col("previous_status") == "Active") &
        (col("status") == "Resigned")
    )
)

print("Resignation / Attrition Events:", resignation_events.count())

display(
    resignation_events.select(
        "emp_id",
        "name",
        "department",
        "previous_status",
        "status",
        "effective_from_day"
    ).orderBy("effective_from_day", "emp_id")
)

In [0]:
total_employees = current_employees.count()
attrition_events = resignation_events.count()

attrition_rate = __builtins__.round(
    (attrition_events / total_employees) * 100,
    2
)

print("Total Current Employees:", total_employees)
print("Attrition Events:", attrition_events)
print("Attrition Rate:", attrition_rate, "%")

In [0]:
attrition_by_department = (
    resignation_events
    .groupBy("department")
    .agg(
        count("*").alias("resignation_count")
    )
    .orderBy(desc("resignation_count"))
)

display(attrition_by_department)

In [0]:
department_attrition = (
    attrition_by_department
    .join(
        department_count,
        on="department",
        how="left"
    )
    .withColumn(
        "attrition_rate",
        round(
            (col("resignation_count") / col("employee_count")) * 100,
            2
        )
    )
    .select(
        "department",
        "employee_count",
        "resignation_count",
        "attrition_rate"
    )
    .orderBy(desc("attrition_rate"))
)

display(department_attrition)

In [0]:
import builtins

average_tenure = overall_tenure.collect()[0]["average_tenure_years"]

total_employees = current_employees.count()
total_resignations = resignation_events.count()

overall_attrition_rate = (
    total_resignations / total_employees
) * 100

highest_salary_department = (
    salary_analysis
    .orderBy(desc("average_salary"))
    .first()["department"]
)

highest_attrition_department = (
    department_attrition
    .orderBy(desc("attrition_rate"))
    .first()["department"]
)

hr_summary = spark.createDataFrame([
    (
        total_employees,
        average_tenure,
        total_resignations,
        builtins.round(overall_attrition_rate, 2),
        highest_salary_department,
        highest_attrition_department
    )
], [
    "current_employees",
    "average_tenure_years",
    "total_resignations",
    "attrition_rate",
    "highest_avg_salary_department",
    "highest_attrition_department"
])

display(hr_summary)

In [0]:
hr_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_hr_summary")

In [0]:
display(
    spark.table("gold_hr_summary")
)

In [0]:
gold_tables = [
    "gold_current_employees",
    "gold_department_salary",
    "gold_department_count",
    "gold_employee_status",
    "gold_department_status",
    "gold_top_salary_employees",
    "gold_hr_summary"
]

for table in gold_tables:
    print(table, "→", spark.table(table).count(), "records")

In [0]:
gold_tables = [
    "gold_current_employees",
    "gold_department_salary",
    "gold_department_count",
    "gold_employee_status",
    "gold_department_status",
    "gold_top_salary_employees",
    "gold_hr_summary"
]

for table in gold_tables:
    print(f"\n{table}")
    print(f"Records: {spark.table(table).count()}")
    spark.table(table).show(5, truncate=False)

In [0]:
display(spark.table("gold_hr_summary"))